In [ ]:
# Import necessary libraries and dependencies
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
# Load the dataset to a pandas DataFrame
zulo_bank = pd.read_csv('dataset/zulo_bank.csv')

In [ ]:
# Display the first 5 rows of the dataset
display(zulo_bank.head(5))

In [ ]:
# Check the data set
zulo_bank.info()

In [ ]:
# Fill up the missing values with the appropriate parameters
zulo_bank.fillna(
    {
        "LoanAmount": 0.0,
        "LoanType": "Unknown",
        "StartDate": "Unknown",
        "EndDate": "Unknown",
        "InterestRate": 0.0,
    },
    inplace=True,
)

In [ ]:
# Check the data set again after filling missing values
zulo_bank.info()

In [ ]:
# Convert to 1NF: First Normal Form
# Spplitting the 'FullName' column into 'first_name' and 'last_name'
zulo_bank[["first_name", "last_name"]] = zulo_bank["FullName"].str.split(expand=True)

In [ ]:
# Converting 1NF to 2NF: Second Normal Form
# Customers table
customers = (
    zulo_bank[["first_name", "last_name", "Email", "Phone"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
customers["customer_id"] = range(1, len(customers) + 1)
customers = customers[["customer_id", "first_name", "last_name", "Email", "Phone"]]

In [ ]:
# Create the Accounts table
accounts = (
    zulo_bank[["AccountType", "Balance", "OpeningDate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
accounts["account_id"] = range(1, len(accounts) + 1)
accounts = accounts[["account_id", "AccountType", "Balance", "OpeningDate"]]

In [ ]:
# Create transactions table
transactions = (
    zulo_bank[["TransactionType", "Amount", "TransactionDate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
transactions["transaction_id"] = range(1, len(transactions) + 1)
transactions = transactions[
    ["transaction_id", "TransactionType", "Amount", "TransactionDate"]
]

In [ ]:
# Create a Loans table
loans = (
    zulo_bank[["LoanType", "LoanAmount", "StartDate", "EndDate", "InterestRate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
loans["loan_id"] = range(1, len(loans) + 1)
loans = loans[
    ["loan_id", "LoanType", "LoanAmount", "StartDate", "EndDate", "InterestRate"]
]

In [ ]:
# Merge operations to create the zulo_bank
zulo_bank = (
    zulo_bank.merge(
        customers, on=["first_name", "last_name", "Email", "Phone"], how="left"
    )
    .merge(accounts, on=["AccountType", "Balance", "OpeningDate"], how="left")
    .merge(
        transactions, on=["TransactionType", "Amount", "TransactionDate"], how="left"
    )
    .merge(
        loans,
        on=["LoanType", "LoanAmount", "StartDate", "EndDate", "InterestRate"],
        how="left",
    )[["customer_id", "account_id", "transaction_id", "loan_id"]]
)

In [ ]:
zulo_bank

In [ ]:
# Convert 2NF to 3NF: Third Normal Form
# Create the date dimension table
# Define the start and end dates for the date dimension
start_date = datetime(2020, 1, 1)
current_date = datetime(2090, 12, 31)

# Calculate the number of days between the start and end dates
num_days = (current_date - start_date).days

# Generate a list of dates from start_date to current_date
date_list = [start_date + timedelta(days=x) for x in range(num_days + 1)]

# ensure date_id matches the length of the date_list
date = {"date_id": [x for x in range(1, len(date_list) + 1)], "date": date_list}

# Create the date dimension DataFrame
date_dim = pd.DataFrame(date)
date_dim["Year"] = date_dim["date"].dt.year
date_dim["Month"] = date_dim["date"].dt.month
date_dim["Day"] = date_dim["date"].dt.day

In [ ]:
date_dim